In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import time
import torch.nn as nn
import torchvision.models as models
from sklearn.metrics import roc_auc_score

In [2]:
SPLITS = "/kaggle/input/datasets/iwmm10/chestxray-capstone-splits"
BASE   = f"{SPLITS}/capstone_splits"
NIH    = "/kaggle/input/datasets/nih-chest-xrays/data"
OUT    = "/kaggle/working"

SPLIT_DIR  = f"{BASE}/splits"
BBOX_CSV   = f"{BASE}/BBox_List_2017.csv"
INDEX_JSON = f"{SPLITS}/image_index.json"

IMG_COL = "Image Index"
PID_COL = "Patient ID"

LABELS = [
    "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
    "Edema", "Emphysema", "Fibrosis", "Effusion",
    "Pneumonia", "Pleural_Thickening", "Cardiomegaly", "Nodule",
    "Mass", "Hernia",
]

SEED     = 42
IMG_SIZE = 224

with open(INDEX_JSON) as f:
    index = json.load(f)

def image_path(fn):
    return f"{NIH}/{index[fn]}/images/{fn}"

print(f"CONFIG OK  |  {len(index):,} images indexed")

CONFIG OK  |  112,120 images indexed


In [3]:
train_df = pd.read_csv(f"{SPLIT_DIR}/train.csv")
val_df   = pd.read_csv(f"{SPLIT_DIR}/val.csv")
test_df  = pd.read_csv(f"{SPLIT_DIR}/test.csv")

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:<7}{len(d):>8,}")

train    18,013
val       3,978
test      3,904


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import time

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomRotation(7),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [6]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, transform):
        self.files  = df[IMG_COL].values
        self.labels = df[LABELS].values.astype("float32")
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        # X-rays are 8-bit grayscale; DenseNet expects 3 channels
        img = Image.open(image_path(self.files[i])).convert("RGB")
        return self.transform(img), torch.from_numpy(self.labels[i])

In [7]:
ds = ChestXrayDataset(train_df, train_tf)
img, lab = ds[0]

print("image :", img.shape, img.dtype)
print("labels:", lab.shape, lab.sum().item(), "positive")
print("range :", f"{img.min():.2f} to {img.max():.2f}")

image : torch.Size([3, 224, 224]) torch.float32
labels: torch.Size([14]) 1.0 positive
range : -2.12 to 2.47


In [8]:
n = 200
start = time.time()
for i in range(n):
    _ = ds[i]
per_img = (time.time() - start) / n

print(f"{per_img*1000:.1f} ms per image (single worker)")
print(f"one epoch on 18,013 images: {per_img*len(train_df)/60:.1f} min")

20.8 ms per image (single worker)
one epoch on 18,013 images: 6.2 min


In [9]:
print("CPU cores:", os.cpu_count())

CPU cores: 4


In [10]:
BATCH_SIZE  = 32
NUM_WORKERS = 4          # adjust if os.cpu_count() is lower

train_loader = DataLoader(
    ChestXrayDataset(train_df, train_tf),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    drop_last=True, persistent_workers=True,
)

val_loader = DataLoader(
    ChestXrayDataset(val_df, eval_tf),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True,
)

xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape)

batch: torch.Size([32, 3, 224, 224]) torch.Size([32, 14])


In [11]:
start = time.time()
for i, _ in enumerate(train_loader):
    if i == 20:
        break
elapsed = time.time() - start

per_img = elapsed / (21 * BATCH_SIZE)
print(f"{per_img*1000:.1f} ms per image with {NUM_WORKERS} workers")
print(f"epoch estimate: {per_img*len(train_df)/60:.1f} min")

15.5 ms per image with 4 workers
epoch estimate: 4.6 min


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def build_model():
    m = models.densenet121(weights="IMAGENET1K_V1")
    m.classifier = nn.Linear(m.classifier.in_features, len(LABELS))
    return m

model = build_model().to(device)
print("outputs:", model.classifier.out_features)

device: cuda | Tesla T4
outputs: 14


In [17]:
# Baseline: plain BCE, no class weighting. This is the reference number that
# later experiments (Focal Loss, pos_weight) must beat to justify themselves.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scaler    = torch.amp.GradScaler('cuda')

EPOCHS = 10

In [18]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, probs, targets = 0.0, [], []

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                out  = model(xb)
                loss = criterion(out, yb)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total += loss.item() * len(xb)
            probs.append(torch.sigmoid(out).detach().float().cpu())
            targets.append(yb.detach().cpu())

    probs   = torch.cat(probs).numpy()
    targets = torch.cat(targets).numpy()

    # Skip classes with no positives in this split — AUROC is undefined there
    aucs = [roc_auc_score(targets[:, i], probs[:, i])
            for i in range(len(LABELS)) if targets[:, i].sum() > 0]

    return total / len(loader.dataset), float(np.mean(aucs))

In [19]:
best = 0.0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc = run_epoch(val_loader,   train=False)

    print(f"epoch {epoch:>2}  "
          f"train {tr_loss:.4f}/{tr_auc:.4f}   "
          f"val {vl_loss:.4f}/{vl_auc:.4f}   "
          f"{time.time()-t0:.0f}s")

    if vl_auc > best:
        best = vl_auc
        torch.save(model.state_dict(), f"{OUT}/baseline_best.pt")
        print(f"          saved (macro AUROC {best:.4f})")

epoch  1  train 0.2527/0.7905   val 0.2583/0.7834   213s
          saved (macro AUROC 0.7834)
epoch  2  train 0.2396/0.8208   val 0.2538/0.7974   199s
          saved (macro AUROC 0.7974)
epoch  3  train 0.2271/0.8469   val 0.2605/0.7927   199s
epoch  4  train 0.2150/0.8670   val 0.2661/0.7912   201s
epoch  5  train 0.2007/0.8888   val 0.2666/0.7939   202s
epoch  6  train 0.1838/0.9106   val 0.2797/0.7835   203s


KeyboardInterrupt: 

In [21]:
import os
print(os.listdir(OUT))
print("size:", os.path.getsize(f"{OUT}/baseline_best.pt")/1e6, "MB")

['baseline_best.pt', '.virtual_documents']
size: 28.482175 MB


In [22]:
from IPython.display import FileLink
FileLink(f"{OUT}/baseline_best.pt")

/kaggle/working/baseline_best.pt